# Sous-échantillonnage du SFT à ~5000 paires

Objectif : réduire l'agrégat SFT réparti (`notebooks/03_repartition_splits.ipynb`, 22407 exemples) à environ 5000 paires, avec `subsample_sft_dataset` (`scripts/extraction.py`).

Le tirage se fait par fraction proportionnelle dans chaque (source, split) : chaque stratum perd la même proportion d'exemples, ce qui préserve à la fois les ratios de splits (80/10/5/5) et les proportions actuelles des sources (donc des langues). Ce choix est documenté dans `docs/decisions.md`, avec un point ouvert à trancher avec le mentor : cette approche garde les proportions naturelles des sources, ce qui donne un dataset à dominante anglaise (MedQuAD), pas un 50/50 fr/en.

In [1]:
import sys
sys.path.append("..")

from collections import Counter
from dotenv import load_dotenv
from scripts.extraction import build_sft_dataset, subsample_sft_dataset, build_sft_sample, TAILLE_CIBLE_SFT

load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Agrégat avant sous-échantillonnage

Rappel de la répartition par source et par split, telle que construite dans le notebook 03.

In [2]:
agregat = build_sft_dataset()
print("agrégat avant sous-échantillonnage :", len(agregat))

sources = sorted(set(r["source"] for r in agregat))
splits = ["train", "validation", "test", "eval_clinique"]
repartition_avant = Counter((r["source"], r["split"]) for r in agregat)

for source in sources:
    total_source = sum(n for (s, _), n in repartition_avant.items() if s == source)
    print(f"{source:15s} {total_source:6d}")

agrégat avant sous-échantillonnage : 22407
frenchmedmcqa     1079
mediqal           4969
medquad          16359


## Sous-échantillonnage

`subsample_sft_dataset` tire dans chaque (source, split) une fraction proportionnelle à sa taille, calculée pour atteindre `TAILLE_CIBLE_SFT` au total.

In [3]:
echantillon = subsample_sft_dataset(agregat)
print("taille cible :", TAILLE_CIBLE_SFT)
print("taille obtenue :", len(echantillon))

taille cible : 5000
taille obtenue : 5001


## Vérification : proportions préservées

Comparaison des proportions par source et par split, avant et après sous-échantillonnage.

In [4]:
repartition_apres = Counter((r["source"], r["split"]) for r in echantillon)

for source in sources:
    total_avant = sum(n for (s, _), n in repartition_avant.items() if s == source)
    total_apres = sum(n for (s, _), n in repartition_apres.items() if s == source)
    print(f"{source:15s} avant {total_avant / len(agregat):.1%}   après {total_apres / len(echantillon):.1%}")

print()
for split in splits:
    total_avant = sum(n for (_, sp), n in repartition_avant.items() if sp == split)
    total_apres = sum(n for (_, sp), n in repartition_apres.items() if sp == split)
    print(f"{split:15s} avant {total_avant / len(agregat):.1%}   après {total_apres / len(echantillon):.1%}")

frenchmedmcqa   avant 4.8%   après 4.8%
mediqal         avant 22.2%   après 22.2%
medquad         avant 73.0%   après 73.0%

train           avant 80.0%   après 80.0%
validation      avant 10.0%   après 10.0%
test            avant 5.0%   après 5.0%
eval_clinique   avant 5.0%   après 5.0%


Les proportions par source et par split restent stables aux arrondis près : le sous-échantillonnage ne déséquilibre ni les langues ni les splits par rapport à l'agrégat de départ.

## Vérification : pas de doublon introduit

Le tirage se fait sans remise à l'intérieur de chaque stratum (`rng.sample`), donc aucun exemple ne peut apparaître deux fois.

In [5]:
ids = [r["id"] for r in echantillon]
print("exemples :", len(ids))
print("ids uniques :", len(set(ids)))

exemples : 5001
ids uniques : 5001


## Détail par (source, split)

In [6]:
for source in sources:
    print(source)
    for split in splits:
        avant = repartition_avant[(source, split)]
        apres = repartition_apres[(source, split)]
        print(f"  {split:15s} {avant:6d} -> {apres:5d}")

frenchmedmcqa
  train              863 ->   193
  validation         108 ->    24
  test                54 ->    12
  eval_clinique       54 ->    12
mediqal
  train             3975 ->   887
  validation         497 ->   111
  test               248 ->    55
  eval_clinique      249 ->    56
medquad
  train            13087 ->  2920
  validation        1636 ->   365
  test               818 ->   183
  eval_clinique      818 ->   183


## Synthèse

Agrégat SFT réduit de 22407 à environ 5000 exemples (`build_sft_sample`), par tirage proportionnel dans chaque (source, split). Les proportions de langues et de splits sont préservées : le dataset final reste à dominante MedQuAD (anglais), reflet direct de la taille de cette source dans le corpus brut.

Point à valider avec le mentor (voir `docs/decisions.md`) : garder cette proportion naturelle (environ 73% anglais / 27% français) ou plafonner MedQuAD pour se rapprocher d'un équilibre 50/50 entre français et anglais.

Prochaine étape : anonymiser SFT et DPO avec Presidio.